# 📈 Stock Analyzer Core - 日本株定量スクリーニング & クオンツ分析基盤

東証全上場約3,900社を対象に、財務諸表（EDINET）と日足市場データを統合し、3層構造（事前足切り・クオンツ配点・ゲートキーパー）によって優良銘柄を高速スクリーニングする実行ノートブックです。

---

### ⚠️ 免責事項（Disclaimer）および利用規約の遵守

> **【投資判断に関する免責事項】**  
> 本ツールおよび解説コンテンツは、公開財務データおよび市場データの自動処理・スクリーニング技術の学習・研究を目的としたものであり、特定の有価証券の売買推奨や投資助言を提供するものではありません。算出されるスコアおよび判定（Verdict）は数式モデルに基づく機械的計算結果であり、将来の運用成果を保証するものではありません。実際の投資判断および売買は、必ずご自身の責任において行ってください。また、本ツールの利用により生じたいかなる損害についても、著者は一切の責任を負いかねます。

> **【データ出典および利用境界】**  
> - **EDINET**: 出典 金融庁 EDINET 閲覧（提出）サイト。本ツールは政府のオープンデータ基本指針に基づき、利用者個人のAPIキーにて通信を行います。
> - **yfinance**: 本ツールは個人学習・研究を目的として yfinance を使用しています。Yahoo Finance の利用規約上の制限やアクセス制限リスク（429 Too Many Requests 等）が存在するため、安定した本番運用や商用利用を行う場合は、日本取引所グループ公式の [J-Quants API](https://jquants.com/) 等の正規データプロバイダの利用を強く推奨します。

## 【Step 0】Google Drive マウント & キャッシュ初期化（Pull Phase）
Google Drive をマウントし、前回のキャッシュデータベースを作業用ローカル超高速SSDへロードします。

In [ ]:
import os
import shutil
from pathlib import Path

# 1. Google Drive のマウント
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("ℹ️ ローカル環境で実行中です。Google Drive マウントをスキップします。")

# 2. パス定数の設定
DRIVE_DIR = Path("/content/drive/MyDrive/StockAnalyzer")
WORKING_DIR = Path("/content/working")

# ディレクトリの作成
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / "cache").mkdir(exist_ok=True)
(DRIVE_DIR / "output").mkdir(exist_ok=True)
(DRIVE_DIR / "config").mkdir(exist_ok=True)

WORKING_DIR.mkdir(parents=True, exist_ok=True)
(WORKING_DIR / "cache").mkdir(exist_ok=True)
(WORKING_DIR / "output").mkdir(exist_ok=True)
(WORKING_DIR / "tmp").mkdir(exist_ok=True)

print("✅ ディレクトリ初期化完了:")
print(f"   永続層 (Google Drive): {DRIVE_DIR}")
print(f"   作業層 (内蔵ローカルSSD): {WORKING_DIR}")

## 【Step 1】環境セットアップ（コード取得 ＆ 依存ライブラリ導入）
公開 GitHub リポジトリから安定版コードを取得し、実行に必要な最小依存関係をインストールします。

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/akkyey/stock-analyzer-core.git"
TARGET_DIR = "/content/stock-analyzer-core"
TARGET_BRANCH = "main"  # 安定タグ発行後は "v1.0.0" 等に切り替え可能

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

subprocess.run(["git", "clone", "--depth", "1", "--branch", TARGET_BRANCH, REPO_URL, TARGET_DIR], check=True)
os.chdir(TARGET_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)

# 既存のキャッシュモジュールを安全に破棄して最新版を確実にロード
for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

# Stage-and-Sync 管理モジュールによる DB ロード (3段構えの復元検証)
from src.utils.colab_sync import ColabSyncManager

if hasattr(ColabSyncManager, "is_first_run"):
    is_first_run = ColabSyncManager.is_first_run(DRIVE_DIR)
else:
    is_first_run = not (Path(DRIVE_DIR) / "stock_analyzer.duckdb").exists()

db_path = ColabSyncManager.pull_database(DRIVE_DIR, WORKING_DIR)
print(f"✅ 実行用 DB 準備完了: {db_path}")

if is_first_run:
    print()
    print("━" * 60)
    print("ℹ️ 【初回実行モードを検知しました】")
    print("   Google Drive に前回のキャッシュが存在しないため、初回全件スキャンを実行します。")
    print("   Step 3 の実行には【約 3〜5 分】かかります。ブラウザを閉じずにお待ちください ☕")
    print("   （※次回以降はキャッシュが効くため、約 1 分 40 秒で完了します）")
    print("━" * 60)
else:
    print()
    print("━" * 60)
    print("⚡ 【差分更新モード（2回目以降）を検知しました】")
    print("   Google Drive から前回のキャッシュを高速ロードしました。")
    print("   Step 3 は新着の差分のみを取得するため、【約 1 分 40 秒】で高速完了します！")
    print("━" * 60)


## 【Step 2】EDINET API キー設定 & 疎通確認
EDINET から有報・四半報を取得するための API キーを設定します。
Colab の「シークレット（左サイドバーの鍵アイコン 🔑）」に `EDINET_API_KEY` を登録するか、以下のセルで対話入力してください。

In [ ]:
import getpass
import os
from datetime import datetime, timedelta

import requests

# 1. API キーの取得
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("EDINET_API_KEY")
except Exception:
    pass

if not api_key:
    api_key = os.environ.get("EDINET_API_KEY")

if not api_key:
    api_key = getpass.getpass("🔑 EDINET APIキーを入力してください (画面には非表示): ")

os.environ["EDINET_API_KEY"] = api_key

# 2. 疎通テスト (ステータスコード & JSON レスポンスの二重判定)
test_date = (datetime.now() - timedelta(days=2)).strftime("%Y-%m-%d")
test_url = f"https://api.edinet-fsa.go.jp/api/v2/documents.json?date={test_date}&type=2&Subscription-Key={api_key}"

try:
    resp = requests.get(test_url, timeout=10)
    if resp.status_code == 401:
        raise ValueError("EDINET APIキーが無効です (401 Unauthorized)。正しいキーを入力してください。")
    data = resp.json()
    if data.get("metadata", {}).get("status") == "401":
        raise ValueError("EDINET APIキーの認証に失敗しました。キーを確認してください。")
    print("✅ EDINET API 疎通確認成功！ 認証は有効です。")
except Exception as e:
    print(f"⚠️ 警告: 疎通テストでエラーが発生しました: {e}")
    print("   APIキーが正しいか確認してください。")

## 【Step 2.5】⚙️ スクリーニング設定のカスタマイズ
ご自身の投資スタイルに合わせて、足切り基準や配点プリセットを調整できます。
※【変更は任意です】標準の黄金比（balanced）でよろしければ、数値を変更せずそのまま再生ボタン（▶）を1回押して通過してください。

In [ ]:
# fmt: off
# @title ⚙️ スクリーニング設定のカスタマイズ { run: "auto" }

# @markdown ### 1. 外部設定ファイル (Drive 上の JSON) の利用
use_custom_json = False  # @param {type:"boolean"} # True の場合、Drive 上の config/custom_config.json を最優先

# @markdown ### 2. 投資スタイル (配点プリセット)
strategy_preset = "balanced"  # @param ["balanced", "dividend_focus", "deep_value", "growth_quality"]

# @markdown ### 3. 第1層 足切り条件 (Pre-Filter)
min_trading_value_million = 30  # @param {type:"slider", min:5, max:100, step:5} # 20日平均売買代金 (百万円)
min_price = 50.0  # @param {type:"number"} # 最低株価 (円)
target_market_ui = "全市場"  # @param ["全市場", "プライムのみ", "スタンダードのみ", "グロースのみ"]

# fmt: on

import json
from pathlib import Path

MARKET_MAP = {
    "全市場": None,
    "プライムのみ": ["Prime"],
    "スタンダードのみ": ["Standard"],
    "グロースのみ": ["Growth"],
}

# 設定の解決
if use_custom_json:
    json_path = DRIVE_DIR / "config" / "custom_config.json"
    if json_path.exists():
        with open(json_path, encoding="utf-8") as f:
            custom_config = json.load(f)
        print(f"✅ Drive 上のカスタム設定を適用しました: {json_path}")
    else:
        print(f"⚠️ {json_path} が見つからないため、Colab 画面の入力値を使用します。")
        use_custom_json = False

if not use_custom_json:
    custom_config = {
        "hard_filters": {
            "min_trading_value": float(min_trading_value_million * 1_000_000),
            "min_price": float(min_price),
            "target_markets": MARKET_MAP.get(target_market_ui, None),
        },
        "strategy_preset": strategy_preset,
        "scoring_multipliers": {},
    }
    print(f"✅ Colab フォームの設定を適用しました: 戦略='{strategy_preset}', 売買代金>={min_trading_value_million}百万円, 最低株価>={min_price}円, 市場='{target_market_ui}'")

## 【Step 3】パイプライン実行 & Drive アトミック同期
作業用超高速SSD上で全銘柄の取得・集計・スクリーニングを実行し、完了後に最新DBと生成レポートを Google Drive へ安全にアトミック同期します。
⏳ **所要時間の目安**: 初回実行時は **約 3〜5 分**、2回目以降は **約 1 分 40 秒** です（ブラウザを閉じずにお待ちください）。

In [ ]:
import os
import time
from pathlib import Path

# 1. 再マウントガード (アンマウント状態であれば自動再マウント)
if not os.path.exists("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass

# 2. 環境変数の設定 (必ず作業用SSDを指す)
os.environ["STOCK_ANALYZER_BASE_DIR"] = "/content/working"

# 3. パイプライン実行
from src.orchestration.context import OrchestratorContext
from src.orchestration.pipeline import OrchestrationPipeline
from src.utils.colab_sync import ColabSyncManager

# 初回実行判定 (メモリ、マネージャ、ファイル直接判定の3段フォールバック)
is_first = globals().get("is_first_run")
if is_first is None:
    if hasattr(ColabSyncManager, "is_first_run"):
        is_first = ColabSyncManager.is_first_run(DRIVE_DIR)
    else:
        drive_path = Path(globals().get("DRIVE_DIR", "/content/drive/MyDrive/StockAnalyzer"))
        is_first = not ((drive_path / "stock_analyzer.duckdb").exists() or (drive_path / "stock_analyzer.duckdb.bak").exists())

if is_first:
    print("⏳ 【初回実行モード】過去30日分の開示書類を初期スキャン中...")
    print("   所要時間は【約 3〜5 分】です。ブラウザを閉じずに少々お待ちください ☕")
else:
    print("⚡ 【差分更新モード】新着決算の差分スキャン中（所要時間: 約 1 分 40 秒）...")
print("🚀 パイプライン実行を開始します...")
start_time = time.time()

context = OrchestratorContext(debug_mode=False)
# カスタム設定の注入 (Step 2.5 未実行時もデフォルトで安全動作)
cfg = globals().get("custom_config", {})
if cfg:
    context.config.update(cfg)

pipeline = OrchestrationPipeline(context)
final_df = pipeline.run()

elapsed = time.time() - start_time
print(f"🎉 全処理が完了しました！ 所要時間: {elapsed:.2f} 秒 ({elapsed / 60:.1f} 分)")

# 4. Google Drive へのアトミック Push & クラウド同期保証 (flush_and_unmount)
print("📦 Google Drive へ最新成果物を同期中...")
sync_ok = ColabSyncManager.push_artifacts(WORKING_DIR, DRIVE_DIR, flush_unmount=True)
if sync_ok:
    print("✨ Google Drive への同期が完了しました (MyDrive/StockAnalyzer/)")
else:
    print("⚠️ Google Drive への同期で一部警告が発生しました。ローカル作業層のファイルを確認してください。")


## 【Step 4】結果プレビュー & サマリ表示
ローカル作業層（`/content/working/output/`）の CSV を読み込み、適格銘柄ランキング Top 10 および除外理由の内訳を表示します。

In [ ]:
from pathlib import Path

import polars as pl

daily_report_path = Path("/content/working/output/daily_report.csv")
uncalculable_path = Path("/content/working/output/uncalculable_stocks.csv")

if daily_report_path.exists():
    df_report = pl.read_csv(daily_report_path)
    print("=======================================================")
    print(f" 🏆 スクリーニング適格銘柄ランキング Top 10 (全 {len(df_report)} 社)")
    print("=======================================================")
    
    display_cols = [c for c in ["Rank", "Code", "Name", "Market", "Verdict", "Agent_Score", "PER", "PBR", "ROE", "Equity_Ratio", "Dividend_Yield"] if c in df_report.columns]
    top10 = df_report.head(10).select(display_cols)
    
    try:
        from IPython.display import display
        display(top10.to_pandas())
    except Exception:
        print(top10)
else:
    print("⚠️ daily_report.csv が見つかりません。Step 3 を実行してください。")

# 除外銘柄の理由別集計
if uncalculable_path.exists():
    df_rej = pl.read_csv(uncalculable_path)
    if "filter_reason" in df_rej.columns:
        reason_counts = df_rej.group_by("filter_reason").len().sort("len", descending=True)
        print(f"\n📊 第1層 足切り除外銘柄の内訳 (全 {len(df_rej)} 社):")
        for row in reason_counts.to_dicts():
            print(f"   ・{row['filter_reason']}: {row['len']} 社")


## 【Step 5】CSV 保存確認 & ローカルダウンロード
生成された `daily_report.csv` を手元の PC へ直接ダウンロードします。

In [ ]:
from pathlib import Path

report_file = "/content/working/output/daily_report.csv"

if Path(report_file).exists():
    try:
        from google.colab import files
        print("📥 daily_report.csv のダウンロードを開始します...")
        files.download(report_file)
    except ImportError:
        print(f"ℹ️ ローカル環境のためブラウザダウンロードをスキップします。ファイルパス: {report_file}")
else:
    print(f"⚠️ ファイルが見つかりません: {report_file}")